# 01 — CPU retrievers: Popularity, ItemCF, ALS and Image

Recommended runtime: G4 High-RAM, CPU only. Prerequisites: Notebook 00 in the same mode, plus the existing five CLIP image artifact files. All four models use the same C0. Popularity is frozen exponentially decayed purchase count. ItemCF retains every historical edge, downweights high-degree users, and multiplies only 32 item columns at a time. ALS uses purchase-count confidence and frozen user factors; missing fitted users fall back to popularity. Image preferences use the latest 100 distinct pre-anchor items, weighted by count and recency. Filename coverage is not independently verified here; existing encoded vectors are reused, never regenerated.

Run only in your own Colab session. This notebook mounts Drive and runs the selected pipeline stage. `MODE="smoke"` is the default; smoke outputs are diagnostics, not full H&M results. Full execution has not been run by the authoring agent. Existing source CSVs and CLIP artifacts are read-only. Metadata/images are static snapshots, not proven historical availability.

## Native Colab dependencies
Run in a fresh runtime. A resolver conflict is an error to investigate; do not remove the PyTorch constraints or reinstall CUDA packages to hide it.

In [1]:
import importlib.metadata
import subprocess
import sys
import tempfile

# Use Colab native Python. Never install the repository's uv environment here.
# Pin any installed PyTorch packages so pip cannot replace Colab CUDA builds.
constraints = []
for package in ("torch", "torchvision", "torchaudio"):
    try:
        constraints.append(f"{package}=={importlib.metadata.version(package)}")
    except importlib.metadata.PackageNotFoundError:
        pass
with tempfile.NamedTemporaryFile(mode="w", suffix=".txt") as handle:
    handle.write("\n".join(constraints))
    handle.flush()
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--quiet",
            "--constraint",
            handle.name,
            *[
                "duckdb==1.5.6",
                "pyarrow>=18,<27",
                "scipy>=1.13,<2",
                "implicit==0.7.3",
                "faiss-cpu>=1.9,<2",
                "threadpoolctl>=3,<4",
            ],
        ],
        check=True,
    )

## Shared protocol and validation contracts
Small helpers are repeated so this notebook has no repository-module dependency.

In [2]:
import hashlib
import json
import math
from datetime import date, timedelta

import numpy as np
import pyarrow as pa

PROTOCOL_VERSION = "1.0"
ARTIFACT_VERSION = "hm-mvp-1.0.0"
FEATURE_SCHEMA_VERSION = "ranking-1.0"
SEED = 20261031
CUTOFF = date(2020, 7, 28)
CHANNELS = ("popularity", "itemcf", "als", "two_tower", "text", "image")
RRF_K0 = 60
CANDIDATES_PER_CHANNEL = 100
FUSED_CANDIDATE_BUDGET = 500
CANDIDATE_SCHEMA = pa.schema(
    [
        ("query_id", pa.string()),
        ("customer_id", pa.string()),
        ("anchor_date", pa.date32()),
        ("article_id", pa.string()),
        ("channel", pa.string()),
        ("raw_score", pa.float32()),
        ("source_rank", pa.int32()),
        ("retriever_version", pa.string()),
    ]
)


def require(condition, message):
    if not condition:
        raise ValueError(message)


def identity(value):
    payload = json.dumps(value, sort_keys=True, separators=(",", ":"), default=str)
    return hashlib.sha256(payload.encode()).hexdigest()


def anchor_schedule():
    starts = [
        ("ranking_train", CUTOFF),
        ("validation", date(2020, 8, 16)),
        ("test", date(2020, 9, 4)),
    ]
    return [
        {
            "split": split,
            "anchor_date": start + timedelta(days=offset),
            "window_end": start + timedelta(days=offset + 7),
        }
        for split, start in starts
        for offset in (0, 3, 6, 9, 12)
    ]


def unique_ranked(items):
    return list(dict.fromkeys(items))


def query_metrics(predictions, truth, k=20):
    truth = set(truth)
    require(bool(truth), "Empty ground truth is not an eligible metric denominator.")
    predictions = unique_ranked(predictions)[:k]
    hits = [int(item in truth) for item in predictions]
    recall = sum(hits) / len(truth)
    dcg = sum(hit / math.log2(j + 2) for j, hit in enumerate(hits))
    ideal = sum(1 / math.log2(j + 2) for j in range(min(k, len(truth))))
    cumulative, precision_sum = 0, 0.0
    for j, hit in enumerate(hits, 1):
        cumulative += hit
        precision_sum += hit * cumulative / j
    return {
        "recall": recall,
        "ndcg": dcg / ideal,
        "ap": precision_sum / min(k, len(truth)),
    }


def stable_top(items, scores, limit=100):
    best = {}
    for item, score in zip(items, scores, strict=True):
        score = float(score)
        require(math.isfinite(score), "Nonfinite retrieval score.")
        best[item] = max(best.get(item, -math.inf), score)
    return sorted(best.items(), key=lambda pair: (-pair[1], pair[0]))[:limit]


def fuse_query(rows, excluded=None, budget=500):
    by_item = {}
    keys = set()
    for row in rows:
        if row["channel"] == excluded:
            continue
        key = row["channel"], row["article_id"]
        require(key not in keys, "Duplicate source candidate before fusion.")
        keys.add(key)
        entry = by_item.setdefault(
            row["article_id"],
            {"article_id": row["article_id"], "rrf": 0.0, "sources": {}},
        )
        entry["sources"][row["channel"]] = (row["source_rank"], row["raw_score"])
        entry["rrf"] += 1.0 / (RRF_K0 + row["source_rank"])
    ordered = sorted(by_item.values(), key=lambda row: (-row["rrf"], row["article_id"]))
    return ordered[:budget], len(ordered)


def validate_candidates(table, roster, catalog, channel):
    require(table.schema.equals(CANDIDATE_SCHEMA), "Candidate schema mismatch.")
    query_lookup = {r["query_id"]: r for r in roster}
    seen, groups = set(), {}
    for row in table.to_pylist():
        qid, item = row["query_id"], row["article_id"]
        require(qid in query_lookup, "Candidate references an unknown query.")
        q = query_lookup[qid]
        require(
            row["customer_id"] == q["customer_id"]
            and row["anchor_date"] == q["anchor_date"],
            "Query identity mismatch.",
        )
        require(item in catalog and isinstance(item, str), "Ineligible candidate ID.")
        require(row["channel"] == channel, "Wrong channel in candidate shard.")
        require(
            row["retriever_version"] == ARTIFACT_VERSION, "Retriever version mismatch."
        )
        require((qid, item) not in seen, "Duplicate query/article candidate.")
        require(math.isfinite(row["raw_score"]), "Nonfinite candidate score.")
        seen.add((qid, item))
        groups.setdefault(qid, []).append(row)
    for rows in groups.values():
        rows.sort(key=lambda r: r["source_rank"])
        require(len(rows) <= 100, "More than 100 candidates for a channel.")
        require(
            [r["source_rank"] for r in rows] == list(range(1, len(rows) + 1)),
            "Source ranks are not contiguous from one.",
        )
        expected = stable_top(
            [r["article_id"] for r in rows], [r["raw_score"] for r in rows]
        )
        require(
            [r["article_id"] for r in rows] == [i for i, _ in expected],
            "Unstable score/article ordering.",
        )
    require(
        set(groups) == set(query_lookup), "A query silently disappeared from a channel."
    )


def validate_mapping(ids, rows):
    require(len(ids) == len(rows), "Mapping length mismatch.")
    require(len(ids) == len(set(ids)), "Duplicate article ID in mapping.")
    require(list(rows) == list(range(len(ids))), "Noncontiguous embedding rows.")
    require(
        all(isinstance(x, str) and len(x) == 10 and x.isdigit() for x in ids),
        "Article IDs must preserve ten-digit strings.",
    )


def validate_manifest(manifest, mode, catalog_id=None):
    fields = {
        "artifact_version",
        "protocol_version",
        "mode",
        "input_paths",
        "input_fingerprints",
        "model_parameters",
        "random_seed",
        "row_count",
        "snapshot_or_anchor",
        "catalog_id",
        "feature_schema_version",
        "output_paths",
        "validation_status",
        "files",
        "stage",
    }
    require(fields <= manifest.keys(), "Incomplete artifact manifest.")
    from pathlib import PurePosixPath

    require(
        set(manifest["files"]) == set(manifest["output_paths"]),
        "Output manifest mismatch.",
    )
    for name in manifest["files"]:
        path = PurePosixPath(name)
        require(
            not path.is_absolute() and ".." not in path.parts and "\\" not in name,
            "Unsafe artifact-relative path.",
        )
    require(
        manifest["artifact_version"] == ARTIFACT_VERSION
        and manifest["protocol_version"] == PROTOCOL_VERSION,
        "Artifact/protocol mismatch.",
    )
    require(manifest["mode"] == mode, "Smoke/full artifact mixing is forbidden.")
    require(manifest["validation_status"] == "validated", "Artifact is not complete.")
    if catalog_id is not None:
        require(manifest["catalog_id"] == catalog_id, "Catalog identity mismatch.")


def metric_smoke_checks():
    result = query_metrics(["a", "x", "b"], {"a", "b"}, 3)
    require(result["recall"] == 1, "Recall formula failed.")
    require(abs(result["ap"] - (1 + 2 / 3) / 2) < 1e-12, "AP formula failed.")
    require(query_metrics([], {"a"})["ndcg"] == 0, "Missing recommendations lost.")
    require(
        query_metrics(["a", "a"], {"a", "b"}, 2)["recall"] == 0.5,
        "Duplicate recommendation received extra credit.",
    )
    schedule = anchor_schedule()
    require(len(schedule) == 15, "Anchor count mismatch.")
    require(schedule[-1]["window_end"] == date(2020, 9, 23), "Test overflow.")

## Runtime, storage and one execution mode
Change only `MODE` between `smoke` and `full`. Outputs use `<stage>/mvp_v1/<mode>/` beneath the specified Drive data root. Completion manifests are written last. Changed inputs or parameters require a new artifact version; completed runs are immutable. Local disk checks include a reserve, but cannot guarantee Colab or Drive quota.

In [3]:
import os
import shutil
import time
import uuid
from pathlib import Path

import duckdb
import pyarrow.parquet as pq

DRIVE_ROOT = Path("/content/drive/MyDrive/projects/e_commerce_recsys/data")
LOCAL_ROOT = Path("/content/data/hm")
WORK_ROOT = Path("/content/recsys_work")
MODE = "full"  # 'smoke' or 'full'; smoke results are diagnostics only.
SMOKE_CUSTOMERS = 2000
SMOKE_QUERIES_PER_ANCHOR = 32
MAX_HISTORY_ITEMS = 100  # Most recent distinct items; query representation only.
RUN_NAMESPACE = "mvp_v1"


def resource_policy(mode, cpus, usable_bytes):
    require(mode in ("smoke", "full"), "MODE must be smoke or full.")
    gib = usable_bytes / 1024**3
    workers = max(1, min(int(cpus), 2 if mode == "smoke" else 16))
    memory = max(0.25, min(4 if mode == "smoke" else 32, gib / 4))
    batch = 256
    if mode == "full":
        batch = 4096 if gib >= 32 else 2048 if gib >= 16 else 512 if gib >= 8 else 256
    return {
        "threads": workers,
        "memory_limit": f"{memory:.2f}GB",
        "query_batch_size": batch,
        "target_shard_bytes": (8 if mode == "smoke" else 192) * 1024**2,
        "als_cache_bytes": int(min(1024**3, usable_bytes * 0.05)),
        "sparse_workspace_bytes": int(min(2 * 1024**3, usable_bytes * 0.04)),
        "usable_ram_bytes": int(usable_bytes),
        "available_cpus": int(cpus),
    }


def detect_resources(proc=Path("/proc"), cgroup=Path("/sys/fs/cgroup")):
    cpu_limits = [os.cpu_count() or 1]
    if hasattr(os, "sched_getaffinity"):
        cpu_limits.append(len(os.sched_getaffinity(0)))
    memory_limits = []
    meminfo = proc / "meminfo"
    if meminfo.is_file():
        values = {
            line.split(":")[0]: int(line.split()[1]) * 1024
            for line in meminfo.read_text().splitlines()
            if line.startswith(("MemAvailable:", "MemTotal:"))
        }
        memory_limits.append(
            values.get("MemAvailable", values.get("MemTotal", 4 * 1024**3))
        )
    directories = {cgroup, cgroup / "memory", cgroup / "cpu", cgroup / "cpu,cpuacct"}
    membership = proc / "self/cgroup"
    if membership.is_file():
        for line in membership.read_text().splitlines():
            _, controllers, relative = line.split(":", 2)
            roots = [cgroup] if not controllers else [cgroup / controllers]
            roots += [cgroup / name for name in controllers.split(",") if name]
            for root in roots:
                child = root / relative.lstrip("/")
                # Include ancestor quotas; a child may say 'max' under a finite parent.
                while child == root or root in child.parents:
                    directories.add(child)
                    if child == root:
                        break
                    child = child.parent
    for directory in directories:
        for limit_name, used_name in [
            ("memory.max", "memory.current"),
            ("memory.limit_in_bytes", "memory.usage_in_bytes"),
        ]:
            limit_path, used_path = directory / limit_name, directory / used_name
            if limit_path.is_file():
                text = limit_path.read_text().strip()
                if text.isdigit() and int(text) < 1 << 60:
                    used = int(used_path.read_text()) if used_path.is_file() else 0
                    memory_limits.append(max(0, int(text) - used))
        quota = directory / "cpu.max"
        if quota.is_file():
            amount, period = quota.read_text().split()
            if amount != "max":
                cpu_limits.append(max(1, int(amount) // int(period)))
        quota, period = directory / "cpu.cfs_quota_us", directory / "cpu.cfs_period_us"
        if quota.is_file() and period.is_file() and int(quota.read_text()) > 0:
            cpu_limits.append(max(1, int(quota.read_text()) // int(period.read_text())))
    # Unknown memory is deliberately conservative; no high-RAM assumption on other OSes.
    return min(cpu_limits), min(memory_limits) if memory_limits else 4 * 1024**3


def configure_resources():
    global THREADS, MEMORY_LIMIT, QUERY_BATCH_SIZE, CPU_RESOURCES
    cpus, memory = detect_resources()
    CPU_RESOURCES = resource_policy(MODE, cpus, memory)
    THREADS = CPU_RESOURCES["threads"]
    MEMORY_LIMIT = CPU_RESOURCES["memory_limit"]
    QUERY_BATCH_SIZE = CPU_RESOURCES["query_batch_size"]
    return CPU_RESOURCES


def rss_gib():
    import resource
    import sys

    peak = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
    return peak / (1024**3 if sys.platform == "darwin" else 1024**2)


def performance_event(operation, started, **fields):
    record = {
        "operation": operation,
        "seconds": time.perf_counter() - started,
        "process_peak_RSS_GiB": rss_gib(),
        **fields,
    }
    CPU_PERFORMANCE.append(record)
    print(json.dumps(record, sort_keys=True))
    return record


CPU_PERFORMANCE = []
CPU_RESOURCES = resource_policy(MODE, *detect_resources())
THREADS = CPU_RESOURCES["threads"]
MEMORY_LIMIT = CPU_RESOURCES["memory_limit"]
QUERY_BATCH_SIZE = CPU_RESOURCES["query_batch_size"]


def sql_string(value):
    return "'" + str(value).replace("'", "''") + "'"


def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(8 * 1024**2), b""):
            h.update(chunk)
    return h.hexdigest()


def disk_guard(path, needed_bytes, operation):
    root = Path(path)
    while not root.exists():
        root = root.parent
    free = shutil.disk_usage(root).free
    require(
        free >= needed_bytes + 2 * 1024**3,
        f"{operation}: need {needed_bytes / 1024**3:.2f} GiB plus 2 GiB reserve; "
        f"only {free / 1024**3:.2f} GiB free at {root}. No population truncated.",
    )


def copy_verified(source, target, expected=None):
    source, target = Path(source), Path(target)
    require(source.is_file(), f"Missing prerequisite file: {source}")
    digest = expected or sha256_file(source)
    if target.is_file() and target.stat().st_size == source.stat().st_size:
        if sha256_file(target) == digest:
            return target
    disk_guard(target.parent, source.stat().st_size, "Stage input")
    target.parent.mkdir(parents=True, exist_ok=True)
    partial = target.with_suffix(target.suffix + ".partial")
    shutil.copyfile(source, partial)
    require(sha256_file(partial) == digest, "Staged file checksum mismatch.")
    os.replace(partial, target)
    return target


def stage_path(stage, root):
    return Path(root) / stage / RUN_NAMESPACE / MODE


def load_stage(stage, catalog_id=None, wanted=None):
    remote = stage_path(stage, DRIVE_ROOT)
    marker = remote / "_COMPLETE.json"
    require(marker.is_file(), f"Complete prerequisite missing: {marker}")
    manifest = json.loads(marker.read_text())
    validate_manifest(manifest, MODE, catalog_id)
    require(manifest["stage"] == stage, "Wrong stage manifest.")
    local = stage_path(stage, WORK_ROOT)
    if local.exists():
        unexpected = [
            p
            for p in local.rglob("*")
            if p.is_file()
            and p.relative_to(local).as_posix() not in manifest["files"]
            and p.name != "_COMPLETE.json"
        ]
        if unexpected:
            local.rename(local.with_name(f"{MODE}.incomplete-{time.time_ns()}"))
    local.mkdir(parents=True, exist_ok=True)
    names = manifest["files"] if wanted is None else wanted
    for name in names:
        require(name in manifest["files"], f"Artifact file not declared: {name}")
        info = manifest["files"][name]
        require(
            (remote / name).stat().st_size == info["bytes"], "Published size mismatch."
        )
        copy_verified(remote / name, local / name, info["sha256"])
    return local, manifest


def reusable(stage, parameters, inputs, catalog_id):
    marker = stage_path(stage, DRIVE_ROOT) / "_COMPLETE.json"
    if not marker.exists():
        return None
    path, manifest = load_stage(stage, catalog_id)
    require(
        manifest["model_parameters"] == parameters,
        "Existing parameters differ; version the run.",
    )
    require(
        manifest["input_fingerprints"] == inputs,
        "Existing inputs differ; version the run.",
    )
    return path, manifest


def runtime_versions():
    from importlib.metadata import PackageNotFoundError, version

    versions = {}
    for name in (
        "duckdb",
        "pyarrow",
        "numpy",
        "scipy",
        "implicit",
        "faiss-cpu",
        "torch",
        "transformers",
        "sentence-transformers",
        "scikit-learn",
        "lightgbm",
        "joblib",
    ):
        try:
            versions[name] = version(name)
        except PackageNotFoundError:
            continue
    return versions


def publish_copy(source, target, expected=None):
    source, target = Path(source), Path(target)
    digest = expected or sha256_file(source)
    if target.exists():
        require(
            target.is_file() and sha256_file(target) == digest,
            f"Conflicting incomplete artifact retained at {target}; "
            "manual review required.",
        )
        return target
    target.parent.mkdir(parents=True, exist_ok=True)
    partial = target.with_name(target.name + "." + uuid.uuid4().hex + ".partial")
    shutil.copyfile(source, partial)
    require(sha256_file(partial) == digest, "Publication checksum mismatch.")
    require(not target.exists(), "Concurrent publication detected; retry verification.")
    # Never rename or delete an existing remote artifact.
    os.rename(partial, target)
    return target


def publish(stage, local, parameters, inputs, catalog_id, row_count, snapshot=None):
    local = Path(local)
    files = {}
    for path in sorted(local.rglob("*")):
        if (
            path.is_file()
            and path.name != "_COMPLETE.json"
            and not path.name.endswith(".partial")
        ):
            name = path.relative_to(local).as_posix()
            files[name] = {"bytes": path.stat().st_size, "sha256": sha256_file(path)}
    require(bool(files), "Refusing to publish an empty stage.")
    manifest = {
        "stage": stage,
        "artifact_version": ARTIFACT_VERSION,
        "protocol_version": PROTOCOL_VERSION,
        "mode": MODE,
        "input_paths": sorted(inputs),
        "input_fingerprints": inputs,
        "model_parameters": parameters,
        "random_seed": SEED,
        "row_count": int(row_count),
        "snapshot_or_anchor": str(snapshot or CUTOFF),
        "catalog_id": catalog_id,
        "feature_schema_version": FEATURE_SCHEMA_VERSION,
        "output_paths": sorted(files),
        "validation_status": "validated",
        "files": files,
        "runtime_versions": runtime_versions(),
        "execution": {"implementation": "cpu-execution-2", "resources": CPU_RESOURCES},
    }
    remote = stage_path(stage, DRIVE_ROOT)
    existing = remote / "_COMPLETE.json"
    if existing.exists():
        require(
            json.loads(existing.read_text()) == manifest,
            "Refusing to replace a completed stage.",
        )
        return manifest
    disk_guard(
        remote, sum(x["bytes"] for x in files.values()), "Publish validated stage"
    )
    remote.mkdir(parents=True, exist_ok=True)
    for name, info in files.items():
        publish_copy(local / name, remote / name, info["sha256"])
    # Completion marker is published last, after every checksum is verified.
    marker = local / "_COMPLETE.json"
    marker.write_text(json.dumps(manifest, indent=2, sort_keys=True))
    publish_copy(marker, existing)
    return manifest


def new_stage(stage):
    path = stage_path(stage, WORK_ROOT)
    if path.exists() and any(path.iterdir()):
        path.rename(path.with_name(f"{MODE}.incomplete-{time.time_ns()}"))
    path.mkdir(parents=True, exist_ok=True)
    # Completion manifests govern reuse of incomplete local runs.
    return path


def write_json(path, value):
    Path(path).write_text(json.dumps(value, indent=2, sort_keys=True, default=str))


def connect_db():
    WORK_ROOT.mkdir(parents=True, exist_ok=True)
    db = duckdb.connect()
    db.execute(f"SET memory_limit={sql_string(MEMORY_LIMIT)}")
    db.execute(f"SET threads={THREADS}")
    db.execute(f"SET temp_directory={sql_string(WORK_ROOT / 'spill')}")
    db.execute("SET preserve_insertion_order=false")
    return db


def export_query(db, query, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    partial = path.with_suffix(".partial")
    db.execute(
        f"COPY ({query}) TO {sql_string(partial)} "
        "(FORMAT PARQUET, COMPRESSION ZSTD, ROW_GROUP_SIZE 65536)"
    )
    os.replace(partial, path)


def view_file(db, name, path):
    db.execute(
        f"CREATE OR REPLACE VIEW {name} AS "
        f"SELECT * FROM read_parquet({sql_string(path)})"
    )


def query_batches(db, anchor):
    previous = ""
    while True:
        rows = db.execute(
            (
                "SELECT query_id, customer_id, anchor_date, split FROM "
                "queries WHERE anchor_date=? AND eligible AND query_id>? "
                "ORDER BY query_id LIMIT ?"
            ),
            [anchor, previous, QUERY_BATCH_SIZE],
        ).fetchall()
        if not rows:
            return
        batch = [
            dict(
                zip(
                    ("query_id", "customer_id", "anchor_date", "split"),
                    row,
                    strict=True,
                )
            )
            for row in rows
        ]
        yield batch
        previous = batch[-1]["query_id"]


def attach_queries(db, batch):
    db.register("batch_queries", pa.Table.from_pylist(batch))


def history_for_batch(db, batch, anchor):
    attach_queries(db, batch)
    # Exact historical counts; only the query preference representation is capped.
    rows = db.execute(
        (
            "WITH grouped AS (\n"
            "        SELECT t.customer_id, t.article_id, count(*) AS n, "
            "max(t.t_dat) AS last_date\n"
            "        FROM transactions t JOIN batch_queries b "
            "USING(customer_id)\n"
            "        WHERE t.t_dat < ? GROUP BY "
            "t.customer_id,t.article_id)\n"
            "        SELECT * FROM grouped QUALIFY row_number() OVER\n"
            "        (PARTITION BY customer_id ORDER BY last_date DESC, "
            "article_id) <= ?\n"
            "        ORDER BY customer_id,last_date DESC,article_id"
        ),
        [anchor, MAX_HISTORY_ITEMS],
    ).fetchall()
    result = {}
    for customer, item, count, last in rows:
        weight = float(count) * math.exp(-(anchor - last).days / 30.0)
        result.setdefault(customer, []).append((item, weight))
    return result


def open_context():
    processed, pm = load_stage(
        "processed",
        wanted=[
            "transactions.parquet",
            "articles.parquet",
            "user_id_map.parquet",
            "article_id_map.parquet",
        ],
    )
    protocol, qm = load_stage("protocol", pm["catalog_id"])
    db = connect_db()
    for name in ("transactions", "articles", "user_id_map", "article_id_map"):
        view_file(db, name, processed / f"{name}.parquet")
    for name in ("eligible_catalog", "anchors"):
        view_file(db, name, protocol / f"{name}.parquet")
    view_file(db, "queries", protocol / "queries" / "*" / "queries.parquet")
    view_file(db, "labels", protocol / "labels" / "*" / "labels.parquet")
    catalog = [
        r[0]
        for r in db.execute(
            "SELECT article_id FROM eligible_catalog ORDER BY article_id"
        ).fetchall()
    ]
    return db, catalog, pm, qm


def runtime_start():
    require(MODE in ("smoke", "full"), "MODE must be smoke or full.")
    from google.colab import drive

    drive.mount("/content/drive")
    WORK_ROOT.mkdir(parents=True, exist_ok=True)
    LOCAL_ROOT.mkdir(parents=True, exist_ok=True)
    np.random.seed(SEED)
    settings = configure_resources()
    print(
        {
            "mode": MODE,
            "resources": settings,
            "result_scope": "diagnostic" if MODE == "smoke" else "full protocol",
            "local_free_GiB": round(shutil.disk_usage(WORK_ROOT).free / 1024**3, 2),
        }
    )

## Bounded graph, index and candidate helpers
Candidate shards are bounded by 256 queries. Every eligible query must appear, and each lane must return at most 100 unique C0 items. Empty history or no model signal uses documented frozen popularity scores. Sparse ItemCF blocks and exact FAISS queries avoid full pairwise matrices. Boundary ties are resolved by article ID.

In [4]:
import pyarrow.compute as pc


def training_graph(db, catalog):
    from scipy import sparse

    users = [
        r[0]
        for r in db.execute(
            "SELECT customer_id FROM user_id_map ORDER BY user_idx"
        ).fetchall()
    ]
    db.register(
        "model_items",
        pa.table({"article_id": catalog, "model_row": range(len(catalog))}),
    )
    edges = db.execute(
        "SELECT u.user_idx AS u,i.model_row AS i,count(*) AS n\n"
        "        FROM transactions t JOIN user_id_map u "
        "USING(customer_id)\n"
        "        JOIN model_items i USING(article_id) WHERE t.t_dat <"
        " DATE '2020-07-28'\n"
        "        GROUP BY u.user_idx,i.model_row ORDER BY "
        "u.user_idx,i.model_row"
    )
    # Stream typed arrays; a complete sparse graph, never a dense user-item matrix.
    u_chunks, i_chunks, n_chunks = [], [], []
    for batch in edges.to_arrow_reader(65536):
        u_chunks.append(batch.column(0).to_numpy().astype(np.int32))
        i_chunks.append(batch.column(1).to_numpy().astype(np.int32))
        n_chunks.append(batch.column(2).to_numpy().astype(np.float32))
    require(bool(u_chunks), "Empty retrieval-training graph.")
    graph = sparse.csr_matrix(
        (
            np.concatenate(n_chunks),
            (np.concatenate(u_chunks), np.concatenate(i_chunks)),
        ),
        shape=(len(users), len(catalog)),
        dtype=np.float32,
    )
    graph.sort_indices()
    return graph, users


def itemcf_neighbors(graph, top_k=64, block_size=None):
    from scipy import sparse

    started = time.perf_counter()
    n_items = graph.shape[1]
    if block_size is None:
        # Conservative dense upper bound for sparse product workspace per row.
        safe_rows = max(
            1, CPU_RESOURCES["sparse_workspace_bytes"] // max(24 * n_items, 1)
        )
        block_size = min(256 if MODE == "full" else 32, safe_rows)
    require(block_size > 0 and top_k > 0, "Invalid sparse block parameters.")
    binary = graph.copy()
    binary.data[:] = 1.0
    degree = np.diff(binary.indptr).astype(np.float32)
    weighted = sparse.diags(1 / np.sqrt(np.maximum(degree, 1))) @ binary
    norms = np.sqrt(np.asarray(weighted.power(2).sum(axis=0)).ravel())
    normalized = (weighted @ sparse.diags(1 / np.maximum(norms, 1e-12))).tocsr()
    columns_out = np.empty(n_items * min(top_k, n_items), dtype=np.int32)
    values_out = np.empty(len(columns_out), dtype=np.float32)
    indptr = np.zeros(n_items + 1, dtype=np.int64)
    cursor = peak_product_bytes = 0
    for start in range(0, n_items, block_size):
        product = (normalized[:, start : start + block_size].T @ normalized).tocsr()
        peak_product_bytes = max(
            peak_product_bytes,
            product.data.nbytes + product.indices.nbytes + product.indptr.nbytes,
        )
        for local_row in range(product.shape[0]):
            a, b = product.indptr[local_row : local_row + 2]
            columns, scores = product.indices[a:b], product.data[a:b]
            keep = (columns != start + local_row) & (scores > 0)
            columns, scores = columns[keep], scores[keep]
            if len(scores) > top_k:
                threshold = np.partition(scores, len(scores) - top_k)[-top_k]
                keep = scores >= threshold
                columns, scores = columns[keep], scores[keep]
            order = np.lexsort((columns, -scores))[:top_k]
            # CSR column ordering matches the original sparse constructor.
            order = order[np.argsort(columns[order])]
            size = len(order)
            columns_out[cursor : cursor + size] = columns[order]
            values_out[cursor : cursor + size] = scores[order]
            cursor += size
            indptr[start + local_row + 1] = cursor
    result = sparse.csr_matrix(
        (values_out[:cursor], columns_out[:cursor], indptr), shape=(n_items, n_items)
    )
    performance_event(
        "itemcf_neighbors",
        started,
        block_size=int(block_size),
        peak_sparse_product_bytes=peak_product_bytes,
        neighbors=result.nnz,
    )
    return result


def flat_index(vectors):
    import faiss

    index = faiss.IndexFlatIP(vectors.shape[1])
    for start in range(0, len(vectors), 4096):
        index.add(np.ascontiguousarray(vectors[start : start + 4096], dtype=np.float32))
    return index


def index_search(index, queries, ids, limit=100):
    queries = np.ascontiguousarray(queries, dtype=np.float32)
    require(len(ids) == index.ntotal, "FAISS row map length mismatch.")
    if not len(ids):
        return [[] for _ in queries]
    scores, positions = index.search(queries, min(limit + 1, len(ids)))
    output = []
    for q, row_scores, row_positions in zip(queries, scores, positions, strict=True):
        valid = row_positions >= 0
        row_scores, row_positions = row_scores[valid], row_positions[valid]
        # Include every boundary tie before lexicographic ID tie-breaking.
        if len(row_scores) > limit and row_scores[limit] == row_scores[limit - 1]:
            threshold = float(
                np.nextafter(np.float32(row_scores[limit - 1]), np.float32(-np.inf))
            )
            _, row_scores, row_positions = index.range_search(q[None, :], threshold)
        output.append(
            stable_top([ids[int(i)] for i in row_positions], row_scores.tolist(), limit)
        )
    return output


def vector_preferences(batch, histories, vectors, id_to_row):
    result = np.zeros((len(batch), vectors.shape[1]), dtype=np.float32)
    for i, query in enumerate(batch):
        history = histories.get(query["customer_id"], [])
        for item, weight in history:
            row = id_to_row.get(item)
            if row is not None:
                result[i] += vectors[row] * weight
    norms = np.linalg.norm(result, axis=1)
    valid = norms > 1e-12
    result[valid] /= norms[valid, None]
    return result, valid


def candidate_stage(split, channel, anchor):
    return f"candidates/{split}/{channel}/{anchor}"


def generate_channel(
    db, catalog, channel, model_manifest, protocol_manifest, retrieve_batch
):
    catalog_set = set(catalog)
    params = {
        "channel": channel,
        "per_channel": 100,
        "query_batch_size": QUERY_BATCH_SIZE,
        "history_item_limit": MAX_HISTORY_ITEMS,
        "history_decay_days": 30,
    }
    for anchor_info in anchor_schedule():
        anchor, split = anchor_info["anchor_date"], anchor_info["split"]
        stage = candidate_stage(split, channel, anchor)
        inputs = {
            "model_manifest": identity(model_manifest),
            "protocol_manifest": identity(protocol_manifest),
        }
        if reusable(stage, params, inputs, identity(catalog)) is not None:
            continue
        local = new_stage(stage)
        count = db.execute(
            "SELECT count(*) FROM queries WHERE eligible AND anchor_date=?", [anchor]
        ).fetchone()[0]
        disk_guard(local, count * 100 * 180, "Candidate anchor output estimate")
        total, fallback_count, seen_queries = 0, 0, 0
        for batch_number, batch in enumerate(query_batches(db, anchor)):
            histories = history_for_batch(db, batch, anchor)
            results, fallbacks = retrieve_batch(batch, histories)
            require(len(results) == len(batch), "Retriever omitted queries.")
            rows = []
            for query, results_for_query in zip(batch, results, strict=True):
                # Sort after float32 conversion, matching persisted score precision.
                ranked = stable_top(
                    [i for i, _ in results_for_query],
                    [float(np.float32(s)) for _, s in results_for_query],
                )
                for rank, (item, score) in enumerate(ranked, 1):
                    rows.append(
                        {
                            **{
                                k: query[k]
                                for k in ("query_id", "customer_id", "anchor_date")
                            },
                            "article_id": item,
                            "channel": channel,
                            "raw_score": score,
                            "source_rank": rank,
                            "retriever_version": ARTIFACT_VERSION,
                        }
                    )
            table = pa.Table.from_pylist(rows, schema=CANDIDATE_SCHEMA)
            validate_candidates(table, batch, catalog_set, channel)
            pq.write_table(
                table, local / f"shard-{batch_number:05d}.parquet", compression="zstd"
            )
            total += len(rows)
            fallback_count += int(fallbacks)
            seen_queries += len(batch)
        require(seen_queries == count, "Full query population was not processed.")
        write_json(
            local / "diagnostics.json",
            {
                "queries": count,
                "candidate_rows": total,
                "fallback_queries": fallback_count,
            },
        )
        publish(stage, local, params, inputs, identity(catalog), total, anchor)
        print(
            {"channel": channel, "anchor": str(anchor), "queries": count, "rows": total}
        )


def content_retriever(vectors, ids, index, fallback):
    id_to_row = {item: i for i, item in enumerate(ids)}

    def retrieve(batch, histories):
        queries, valid = vector_preferences(batch, histories, vectors, id_to_row)
        results = index_search(index, queries, ids)
        for i in np.flatnonzero(~valid):
            results[int(i)] = fallback
        return results, int((~valid).sum())

    return retrieve


def cpu_candidate_parameters(channel):
    # Legacy compatibility descriptor, NOT the actual optimized execution settings.
    # Reader 03 compares this dict across anchors. Actual settings live in execution.
    result = {
        "channel": channel,
        "per_channel": 100,
        "query_batch_size": 256,
        "history_item_limit": MAX_HISTORY_ITEMS,
        "history_decay_days": 30,
    }
    discovered = None
    for info in anchor_schedule():
        stage = candidate_stage(info["split"], channel, info["anchor_date"])
        path = stage_path(stage, DRIVE_ROOT) / "_COMPLETE.json"
        if not path.exists():
            continue
        manifest = json.loads(path.read_text())
        validate_manifest(manifest, MODE)
        params = manifest["model_parameters"]
        require(
            all(
                params.get(k) == v for k, v in result.items() if k != "query_batch_size"
            ),
            "Completed candidate semantics differ; explicit migration required.",
        )
        require(
            set(params) == set(result)
            and isinstance(params["query_batch_size"], int)
            and params["query_batch_size"] > 0,
            "Unknown candidate compatibility descriptor.",
        )
        if discovered is not None:
            require(
                params == discovered, "Existing anchors have incompatible descriptors."
            )
        discovered = params
    return discovered or result


def candidate_reuse(db, stage, parameters, inputs, catalog_id, anchor, roster):
    path = stage_path(stage, DRIVE_ROOT)
    marker = path / "_COMPLETE.json"
    if not marker.exists():
        return None
    started = time.perf_counter()
    manifest = json.loads(marker.read_text())
    validate_manifest(manifest, MODE, catalog_id)
    require(
        manifest["stage"] == stage and manifest["snapshot_or_anchor"] == str(anchor),
        "Completed candidate stage identity mismatch.",
    )
    require(
        manifest["model_parameters"] == parameters
        and manifest["input_fingerprints"] == inputs,
        "Completed candidates are incompatible; not overwritten.",
    )
    rows = 0
    files = []
    for name, info in manifest["files"].items():
        file = path / name
        require(
            file.is_file() and file.stat().st_size == info["bytes"],
            "Published size mismatch.",
        )
        require(sha256_file(file) == info["sha256"], "Published checksum mismatch.")
        if name.startswith("shard-") and name.endswith(".parquet"):
            parquet = pq.ParquetFile(file)
            require(
                parquet.schema_arrow.remove_metadata().equals(CANDIDATE_SCHEMA),
                "Completed candidate schema mismatch.",
            )
            rows += parquet.metadata.num_rows
            files.append(file)
    require(files and rows == manifest["row_count"], "Completed row totals mismatch.")
    # Prior validation_status plus checked bytes preserve the original validation proof.
    require(
        len(roster) <= rows <= len(roster) * 100,
        "Completed query/row counts impossible.",
    )
    if parameters["channel"] in ("popularity", "als"):
        require(
            rows
            == len(roster)
            * min(
                100, db.execute("SELECT count(*) FROM eligible_catalog").fetchone()[0]
            ),
            "Completed fixed-length candidate count mismatch.",
        )
    performance_event(
        "verified_completed_anchor",
        started,
        stage=stage,
        rows=rows,
        files=len(files),
        copied_candidate_files=0,
    )
    return manifest


def anchor_roster(db, anchor):
    table = db.execute(
        """SELECT query_id,customer_id,anchor_date FROM queries
        WHERE eligible AND anchor_date=? ORDER BY query_id""",
        [anchor],
    ).to_arrow_table()
    require(len(table) > 0, "No eligible queries; no population silently skipped.")
    require(len(pc.unique(table["query_id"])) == len(table), "Duplicate query IDs.")
    require(
        len(pc.unique(table["customer_id"])) == len(table),
        "Repeated customer within anchor.",
    )
    return table


def prepare_anchor_history(db, roster, anchor):
    started = time.perf_counter()
    db.register(
        "cpu_anchor_queries",
        roster.append_column(
            "query_row", pa.array(np.arange(len(roster), dtype=np.int32))
        ),
    )
    table = db.execute(
        """WITH grouped AS (
        SELECT q.query_row,t.article_id,count(*) AS n,max(t.t_dat) AS last_date
        FROM transactions t JOIN cpu_anchor_queries q USING(customer_id)
        WHERE t.t_dat < ? GROUP BY q.query_row,t.article_id),
        limited AS (SELECT * FROM grouped QUALIFY row_number() OVER
          (PARTITION BY query_row ORDER BY last_date DESC,article_id) <= ?)
        SELECT l.query_row,m.item_idx,l.n,date_diff('day',l.last_date,?) AS age
        FROM limited l JOIN article_id_map m USING(article_id)
        ORDER BY l.query_row,l.last_date DESC,l.article_id""",
        [anchor, MAX_HISTORY_ITEMS, anchor],
    ).to_arrow_table()
    qrows = table["query_row"].to_numpy().astype(np.int32)
    items = table["item_idx"].to_numpy().astype(np.int32)
    counts = table["n"].to_numpy().astype(np.float64)
    ages = table["age"].to_numpy()
    unique_ages, inverse = np.unique(ages, return_inverse=True)
    # Scalar math.exp matches the prior float64 history-weight calculation exactly.
    decay = np.fromiter(
        (math.exp(-int(age) / 30.0) for age in unique_ages), dtype=np.float64
    )
    weights = counts * decay[inverse]
    indptr = np.concatenate(
        ([0], np.cumsum(np.bincount(qrows, minlength=len(roster))))
    ).astype(np.int64)
    require(
        np.diff(indptr).max(initial=0) <= MAX_HISTORY_ITEMS, "History limit changed."
    )
    history = {"indptr": indptr, "items": items, "weights": weights}
    record = performance_event(
        "anchor_history",
        started,
        anchor=str(anchor),
        queries=len(roster),
        history_rows=len(items),
        array_bytes=sum(v.nbytes for v in history.values()),
    )
    history["seconds"] = record["seconds"]
    return history


def source_item_mapping(db, model_ids):
    mapping = db.execute(
        "SELECT article_id,item_idx FROM article_id_map ORDER BY item_idx"
    ).to_arrow_table()
    require(
        np.array_equal(mapping["item_idx"].to_numpy(), np.arange(len(mapping))),
        "Canonical item mapping is not contiguous.",
    )
    positions = pc.index_in(mapping["article_id"], value_set=pa.array(model_ids))
    return pc.fill_null(positions, -1).to_numpy().astype(np.int32)


def empty_results(n):
    return (
        np.full((n, 100), -1, dtype=np.int32),
        np.zeros((n, 100), dtype=np.float32),
        np.zeros(n, dtype=np.int16),
    )


def store_ranked(target, row, positions, scores, fallback):
    ids, values, lengths = target
    if len(positions) == 0:
        positions, scores = fallback
    # The caller selected Top-100 using original precision; persist/order float32.
    scores = np.asarray(scores, dtype=np.float32)
    positions = np.asarray(positions, dtype=np.int32)
    order = np.lexsort((positions, -scores))
    n = len(order)
    require(0 < n <= 100, "Invalid recommendation count.")
    ids[row, :n], values[row, :n], lengths[row] = positions[order], scores[order], n


def compact_cf_results(history, start, end, source_to_model, neighbors, fallback):
    result = empty_results(end - start)
    fallback_count = 0
    for output_row, query_row in enumerate(range(start, end)):
        lo, hi = history["indptr"][query_row : query_row + 2]
        model_rows = source_to_model[history["items"][lo:hi]]
        valid = model_rows >= 0
        model_rows, weights = model_rows[valid], history["weights"][lo:hi][valid]
        starts, ends = neighbors.indptr[model_rows], neighbors.indptr[model_rows + 1]
        counts = ends - starts
        total = int(counts.sum())
        if total:
            offsets = np.repeat(starts - np.cumsum(counts) + counts, counts)
            locations = offsets + np.arange(total)
            columns = neighbors.indices[locations]
            contributions = neighbors.data[locations].astype(np.float64) * np.repeat(
                weights, counts
            )
            unique, inverse = np.unique(columns, return_inverse=True)
            # bincount accumulates in original history order, like the Python float sum.
            scores = np.bincount(inverse, weights=contributions, minlength=len(unique))
            order = np.lexsort((unique, -scores))[:100]
            positions, scores = unique[order], scores[order]
        else:
            positions, scores = [], []
            fallback_count += 1
        store_ranked(result, output_row, positions, scores, fallback)
    return result, fallback_count


def compact_image_preferences(history, start, end, source_to_model, vectors):
    result = np.zeros((end - start, vectors.shape[1]), dtype=np.float32)
    bases = history["indptr"][start:end]
    lengths = np.diff(history["indptr"][start : end + 1])
    # At each history position, update different queries in parallel. Sequential
    # position order and float32 multiply/add match the legacy preference reduction.
    for offset in range(int(lengths.max(initial=0))):
        query_rows = np.flatnonzero(lengths > offset)
        locations = bases[query_rows] + offset
        rows = source_to_model[history["items"][locations]]
        valid = rows >= 0
        query_rows, locations, rows = query_rows[valid], locations[valid], rows[valid]
        if len(rows):
            result[query_rows] += (
                vectors[rows]
                * history["weights"][locations].astype(np.float32)[:, None]
            )
    norms = np.linalg.norm(result, axis=1)
    valid = norms > 1e-12
    result[valid] /= norms[valid, None]
    return result, valid


def compact_index_results(
    index, vectors, valid, model_ids, catalog_positions, fallback, model_order=None
):
    result = empty_results(len(vectors))
    active = np.flatnonzero(valid)
    for start in range(0, len(active), 256):
        # Keep legacy-sized FAISS search blocks; no approximate index is introduced.
        rows = active[start : start + 256]
        ranked = index_search(index, vectors[rows], model_ids)
        for row, pairs in zip(rows, ranked, strict=True):
            ids = [item for item, _ in pairs]
            positions = np.searchsorted(model_ids, ids, sorter=model_order)
            if model_order is not None:
                positions = model_order[positions]
            store_ranked(
                result,
                int(row),
                catalog_positions[positions],
                [s for _, s in pairs],
                fallback,
            )
    for row in np.flatnonzero(~valid):
        store_ranked(result, int(row), [], [], fallback)
    return result, int((~valid).sum())


def candidate_arrow(roster, result, catalog, channel):
    ids, scores, lengths = result
    require(len(roster) == len(lengths), "Retriever omitted a query.")
    require(np.all((lengths > 0) & (lengths <= 100)), "Invalid candidate lengths.")
    mask = np.arange(100)[None, :] < lengths[:, None]
    item_rows = ids[mask]
    require(
        np.all((item_rows >= 0) & (item_rows < len(catalog))), "Ineligible item row."
    )
    query_rows = pa.array(np.repeat(np.arange(len(roster), dtype=np.int32), lengths))
    n = len(item_rows)
    zeros = pa.array(np.zeros(n, dtype=np.int8))
    return pa.Table.from_arrays(
        [
            pc.take(roster["query_id"], query_rows),
            pc.take(roster["customer_id"], query_rows),
            pc.take(roster["anchor_date"], query_rows),
            pc.take(catalog, pa.array(item_rows)),
            pc.take(pa.array([channel]), zeros),
            pa.array(scores[mask], type=pa.float32()),
            pa.array(
                np.broadcast_to(np.arange(1, 101, dtype=np.int32), mask.shape)[mask]
            ),
            pc.take(pa.array([ARTIFACT_VERSION]), zeros),
        ],
        schema=CANDIDATE_SCHEMA,
    )


class CandidateWriter:
    def __init__(self, local, attempt):
        self.local = Path(local)
        self.attempt = attempt
        self.writer = None
        self.path = None
        self.files = []
        self.seconds = 0.0
        self.rows = 0

    def write(self, table):
        started = time.perf_counter()
        require(
            table.schema.remove_metadata().equals(CANDIDATE_SCHEMA),
            "Output schema mismatch.",
        )
        if self.writer is None:
            self.path = (
                self.local / f"shard-{self.attempt}-{len(self.files):05d}.parquet"
            )
            self.writer = pq.ParquetWriter(
                self.path, CANDIDATE_SCHEMA, compression="zstd", use_dictionary=True
            )
            self.files.append(self.path)
        self.writer.write_table(table, row_group_size=131072)
        self.rows += len(table)
        if self.path.stat().st_size >= CPU_RESOURCES["target_shard_bytes"]:
            self.writer.close()
            self.writer = None
        self.seconds += time.perf_counter() - started

    def close(self):
        if self.writer is not None:
            started = time.perf_counter()
            self.writer.close()
            self.writer = None
            self.seconds += time.perf_counter() - started


def validate_candidate_files(db, files, roster, catalog, channel):
    require(bool(files), "No candidate shards produced.")
    total = 0
    for path in files:
        file = pq.ParquetFile(path)
        require(
            file.schema_arrow.remove_metadata().equals(CANDIDATE_SCHEMA),
            "Candidate schema mismatch.",
        )
        total += file.metadata.num_rows
    db.register("cpu_validation_queries", roster)
    db.register("cpu_validation_catalog", pa.table({"article_id": catalog}))
    paths = "[" + ",".join(sql_string(p) for p in files) + "]"
    db.execute(
        f"CREATE OR REPLACE VIEW cpu_validate AS SELECT * FROM read_parquet({paths})"
    )
    bad = db.execute(
        """SELECT count(*) FROM cpu_validate c
        LEFT JOIN cpu_validation_queries q USING(query_id)
        WHERE q.query_id IS NULL OR c.customer_id IS DISTINCT FROM q.customer_id
        OR c.anchor_date IS DISTINCT FROM q.anchor_date
        OR c.channel IS DISTINCT FROM ? OR c.retriever_version IS DISTINCT FROM ?
        OR c.raw_score IS NULL OR NOT isfinite(c.raw_score)
        OR c.source_rank IS NULL OR c.article_id IS NULL
        OR NOT EXISTS (SELECT 1 FROM cpu_validation_catalog a
        WHERE a.article_id=c.article_id)""",
        [channel, ARTIFACT_VERSION],
    ).fetchone()[0]
    require(bad == 0, "Candidate IDs, finite scores or query metadata invalid.")
    invalid_groups = db.execute("""SELECT count(*) FROM (
        SELECT query_id FROM cpu_validate GROUP BY query_id
        HAVING count(*)>100 OR count(DISTINCT article_id)<>count(*)
        OR min(source_rank)<>1 OR max(source_rank)<>count(*)
        OR count(DISTINCT source_rank)<>count(*))""").fetchone()[0]
    require(invalid_groups == 0, "Candidate uniqueness/rank contract failed.")
    queries = db.execute(
        "SELECT count(DISTINCT query_id) FROM cpu_validate"
    ).fetchone()[0]
    require(queries == len(roster), "Eligible query coverage changed.")
    unstable = db.execute("""SELECT count(*) FROM (
        SELECT source_rank,row_number() OVER (PARTITION BY query_id
        ORDER BY raw_score DESC,article_id) AS expected FROM cpu_validate)
        WHERE source_rank<>expected""").fetchone()[0]
    require(unstable == 0, "Unstable source score/rank ordering.")
    return total


def generate_popularity_anchor(db, roster, fallback, writer):
    db.register("cpu_pop_queries", roster)
    ids, scores = zip(*fallback, strict=True)
    # Scores already have their persisted float32 precision; rank ties by article ID.
    top = pa.table(
        {
            "article_id": list(ids),
            "score": pa.array(scores, type=pa.float32()),
            "source_rank": pa.array(range(1, len(ids) + 1), type=pa.int32()),
        }
    )
    db.register("cpu_pop_top100", top)
    started = time.perf_counter()
    reader = db.execute(
        """SELECT q.query_id,q.customer_id,q.anchor_date,p.article_id,
        'popularity'::VARCHAR AS channel,p.score::FLOAT AS raw_score,
        p.source_rank::INTEGER AS source_rank,?::VARCHAR AS retriever_version
        FROM cpu_pop_queries q CROSS JOIN cpu_pop_top100 p
        ORDER BY q.query_id,p.source_rank""",
        [ARTIFACT_VERSION],
    ).to_arrow_reader(QUERY_BATCH_SIZE * 100)
    compute = time.perf_counter() - started
    while True:
        started = time.perf_counter()
        batch = next(reader, None)
        compute += time.perf_counter() - started
        if batch is None:
            break
        writer.write(pa.Table.from_batches([batch]))
    return compute, 0


def prepare_frozen_als(db, local, catalog, fallback):
    users = pq.read_table(local / "users.parquet")["customer_id"].combine_chunks()
    ids = pq.read_table(local / "items.parquet")["article_id"].to_pylist()
    require(
        ids == catalog and len(pc.unique(users)) == len(users),
        "ALS ID mapping mismatch.",
    )
    user_factors = np.load(local / "user_factors.npy", mmap_mode="r")
    item_factors = np.load(local / "item_factors.npy", mmap_mode="r")
    has_history = np.load(local / "has_history.npy", mmap_mode="r")
    require(
        user_factors.shape == (len(users), 64) and item_factors.shape == (len(ids), 64),
        "ALS factor shape mismatch.",
    )
    require(
        np.isfinite(user_factors).all() and np.isfinite(item_factors).all(),
        "Nonfinite ALS factors.",
    )
    if MODE == "full" and user_factors.nbytes < CPU_RESOURCES["usable_ram_bytes"] / 10:
        user_factors = np.asarray(user_factors).copy()
    queried = db.execute("""SELECT customer_id,count(*) AS anchors FROM queries
        WHERE eligible GROUP BY customer_id ORDER BY customer_id""").to_arrow_table()
    mapped = (
        pc.fill_null(pc.index_in(queried["customer_id"], value_set=users), -1)
        .to_numpy()
        .astype(np.int32)
    )
    counts = queried["anchors"].to_numpy()
    valid = mapped >= 0
    valid[valid] &= has_history[mapped[valid]]
    capacity = CPU_RESOURCES["als_cache_bytes"] // (100 * 8 + 3)
    repeated = np.flatnonzero(valid & (counts > 1))
    order = np.lexsort((mapped[repeated], -counts[repeated]))[:capacity]
    selected = mapped[repeated[order]]
    slots = np.full(len(users), -1, dtype=np.int32)
    slots[selected] = np.arange(len(selected), dtype=np.int32)
    result = {
        "query_ids": queried["customer_id"],
        "model_rows": mapped,
        "has_history": has_history,
        "factors": user_factors,
        "index": flat_index(item_factors),
        "ids": ids,
        "slots": slots,
        "cached": empty_results(len(selected)),
        "ready": np.zeros(len(selected), dtype=bool),
        "fallback": fallback,
        "hits": 0,
        "misses": 0,
        "search_seconds": 0.0,
    }
    print(
        json.dumps(
            {
                "als_cache_users": len(selected),
                "eligible_unique_users": len(mapped),
                "cache_bytes": sum(a.nbytes for a in result["cached"]),
                "policy": "most repeated eligible fitted users; bounded capacity",
            }
        )
    )
    return result


def als_anchor_rows(cache, roster):
    lookup = pc.index_in(roster["customer_id"], value_set=cache["query_ids"])
    require(lookup.null_count == 0, "Unknown eligible customer in ALS lookup.")
    return cache["model_rows"][lookup.to_numpy()]


def frozen_als_results(cache, model_rows):
    output = empty_results(len(model_rows))
    pending, cached_rows, fallback_rows = [], [], []
    for row, model_row in enumerate(model_rows):
        if model_row < 0 or not cache["has_history"][model_row]:
            fallback_rows.append(row)
        else:
            slot = cache["slots"][model_row]
            if slot >= 0 and cache["ready"][slot]:
                cached_rows.append((row, slot))
            else:
                pending.append(row)
    for row, slot in cached_rows:
        for target, source in zip(output, cache["cached"], strict=True):
            target[row] = source[slot]
    if pending:
        started = time.perf_counter()
        vectors = np.asarray(cache["factors"][model_rows[pending]], dtype=np.float32)
        fresh, _ = compact_index_results(
            cache["index"],
            vectors,
            np.ones(len(pending), dtype=bool),
            cache["ids"],
            np.arange(len(cache["ids"]), dtype=np.int32),
            cache["fallback"],
        )
        cache["search_seconds"] += time.perf_counter() - started
        for offset, row in enumerate(pending):
            for target, source in zip(output, fresh, strict=True):
                target[row] = source[offset]
            slot = cache["slots"][model_rows[row]]
            if slot >= 0:
                for target, source in zip(cache["cached"], fresh, strict=True):
                    target[slot] = source[offset]
                cache["ready"][slot] = True
    for row in fallback_rows:
        store_ranked(output, row, [], [], cache["fallback"])
    cache["hits"] += len(cached_rows)
    cache["misses"] += len(pending)
    return output, len(fallback_rows)


def emit_cpu_anchor(
    db,
    roster,
    catalog,
    channel,
    anchor,
    split,
    model_manifest,
    protocol_manifest,
    parameters,
    compute=None,
    popularity=None,
    history_seconds=0.0,
):
    stage = candidate_stage(split, channel, anchor)
    inputs = {
        "model_manifest": identity(model_manifest),
        "protocol_manifest": identity(protocol_manifest),
    }
    local = new_stage(stage)
    disk_guard(local, len(roster) * 100 * 180, "Candidate anchor output estimate")
    attempt = "cpu2-" + uuid.uuid4().hex[:12]
    writer = CandidateWriter(local, attempt)
    compute_seconds, fallback_count = 0.0, 0
    try:
        if channel == "popularity":
            compute_seconds, fallback_count = generate_popularity_anchor(
                db, roster, popularity, writer
            )
        else:
            catalog_arrow = pa.array(catalog)
            for start in range(0, len(roster), QUERY_BATCH_SIZE):
                end = min(start + QUERY_BATCH_SIZE, len(roster))
                started = time.perf_counter()
                result, missing = compute(start, end)
                table = candidate_arrow(
                    roster.slice(start, end - start), result, catalog_arrow, channel
                )
                compute_seconds += time.perf_counter() - started
                fallback_count += missing
                writer.write(table)
    finally:
        writer.close()
    validation_started = time.perf_counter()
    rows = validate_candidate_files(db, writer.files, roster, catalog, channel)
    require(rows == writer.rows, "Writer row count mismatch.")
    validation_seconds = time.perf_counter() - validation_started
    total_bytes = sum(path.stat().st_size for path in writer.files)
    diagnostics = {
        "channel": channel,
        "anchor": str(anchor),
        "queries": len(roster),
        "candidate_rows": rows,
        "fallback_queries": fallback_count,
        "history_seconds": history_seconds,
        "compute_seconds": compute_seconds,
        "write_seconds": writer.seconds,
        "validation_seconds": validation_seconds,
        "rows_per_compute_write_second": rows
        / max(compute_seconds + writer.seconds, 1e-9),
        "parquet_files": len(writer.files),
        "parquet_bytes": total_bytes,
        "process_peak_RSS_GiB": rss_gib(),
    }
    write_json(local / f"diagnostics-{attempt}.json", diagnostics)
    started = time.perf_counter()
    publish(stage, local, parameters, inputs, identity(catalog), rows, anchor)
    diagnostics["publish_seconds"] = time.perf_counter() - started
    diagnostics["rows_per_total_second"] = rows / max(
        compute_seconds
        + writer.seconds
        + validation_seconds
        + diagnostics["publish_seconds"],
        1e-9,
    )
    CPU_PERFORMANCE.append(diagnostics)
    print(json.dumps(diagnostics, sort_keys=True))
    return diagnostics


def pending_cpu_anchor(
    db, roster, catalog, channel, info, manifest, protocol, parameters
):
    stage = candidate_stage(info["split"], channel, info["anchor_date"])
    inputs = {
        "model_manifest": identity(manifest),
        "protocol_manifest": identity(protocol),
    }
    return (
        candidate_reuse(
            db,
            stage,
            parameters,
            inputs,
            identity(catalog),
            info["anchor_date"],
            roster,
        )
        is None
    )

## Four frozen retrieval implementations
Fit once before 2020-07-28, save and verify reloads, then generate candidates by anchor. Image vectors and every FAISS reconstructed row are validated. Missing image articles remain eligible in C0 and other lanes.

In [5]:
from functools import partial


def fit_cpu_models(db, catalog, processed_manifest):
    from scipy import sparse

    inputs = {"processed_manifest": identity(processed_manifest)}
    catalog_id = identity(catalog)
    popularity_params = {"cutoff": str(CUTOFF), "decay_days": 30.0}
    reused = reusable("retrievers/popularity", popularity_params, inputs, catalog_id)
    if reused is None:
        local = new_stage("retrievers/popularity")
        export_query(
            db,
            (
                "SELECT article_id,\n"
                "            sum(exp(-date_diff('day',t_dat,DATE "
                "'2020-07-28')/30.0))::FLOAT AS score,\n"
                "            count(*) AS purchase_lines FROM transactions\n"
                "            WHERE t_dat < DATE '2020-07-28' AND article_id "
                "IN\n"
                "            (SELECT article_id FROM eligible_catalog)\n"
                "            GROUP BY article_id ORDER BY score "
                "DESC,article_id"
            ),
            local / "popularity.parquet",
        )
        pm = publish(
            "retrievers/popularity",
            local,
            popularity_params,
            inputs,
            catalog_id,
            len(catalog),
        )
    else:
        local, pm = reused
    fallback = db.execute(
        "SELECT article_id,score FROM read_parquet(?) "
        "ORDER BY score DESC,article_id LIMIT 100",
        [str(local / "popularity.parquet")],
    ).fetchall()
    require(len(fallback) == min(100, len(catalog)), "Incomplete popularity fallback.")
    cf_params = {
        "cutoff": str(CUTOFF),
        "top_k": 64,
        "block_size": 32,
        "graph": "all edges; binary; inverse-sqrt user-degree weighting; cosine",
    }
    als_params = {
        "cutoff": str(CUTOFF),
        "factors": 64,
        "regularization": 0.05,
        "iterations": 2 if MODE == "smoke" else 15,
        "confidence": "1 + 40 * log1p(purchase_lines)",
        "seed": SEED,
    }
    cf_reuse = reusable("retrievers/itemcf", cf_params, inputs, catalog_id)
    als_reuse = reusable("retrievers/als", als_params, inputs, catalog_id)
    if cf_reuse is None or als_reuse is None:
        graph_started = time.perf_counter()
        graph, users = training_graph(db, catalog)
        performance_event("training_graph", graph_started, edges=graph.nnz)
        disk_guard(
            WORK_ROOT, graph.data.nbytes * 12 + 1024**3, "Sparse model artifacts/spill"
        )
        if cf_reuse is None:
            local = new_stage("retrievers/itemcf")
            neighbors = itemcf_neighbors(graph)
            sparse.save_npz(local / "neighbors.npz", neighbors)
            pq.write_table(pa.table({"article_id": catalog}), local / "items.parquet")
            loaded = sparse.load_npz(local / "neighbors.npz")
            require((loaded != neighbors).nnz == 0, "ItemCF reload mismatch.")
            cm = publish(
                "retrievers/itemcf", local, cf_params, inputs, catalog_id, neighbors.nnz
            )
            cf_reuse = local, cm
        if als_reuse is None:
            from implicit.cpu.als import AlternatingLeastSquares
            from threadpoolctl import threadpool_limits

            local = new_stage("retrievers/als")
            confidence = graph.copy()
            confidence.data = 1.0 + 40.0 * np.log1p(confidence.data)
            model = AlternatingLeastSquares(
                factors=64,
                regularization=0.05,
                iterations=als_params["iterations"],
                random_state=SEED,
                num_threads=THREADS,
            )
            fit_started = time.perf_counter()
            with threadpool_limits(limits=1, user_api="blas"):
                model.fit(confidence, show_progress=True)
            performance_event("als_fit", fit_started, threads=THREADS, blas_threads=1)
            np.save(local / "user_factors.npy", model.user_factors)
            np.save(local / "item_factors.npy", model.item_factors)
            np.save(local / "has_history.npy", np.diff(graph.indptr) > 0)
            pq.write_table(pa.table({"customer_id": users}), local / "users.parquet")
            pq.write_table(pa.table({"article_id": catalog}), local / "items.parquet")
            require(
                np.array_equal(
                    np.load(local / "user_factors.npy", mmap_mode="r"),
                    model.user_factors,
                ),
                "ALS user factor reload mismatch.",
            )
            am = publish(
                "retrievers/als", local, als_params, inputs, catalog_id, graph.nnz
            )
            als_reuse = local, am
            del confidence, model
        del graph, users
    return fallback, pm, cf_reuse, als_reuse


def prepare_image_model(db, catalog):
    import faiss

    source = DRIVE_ROOT / "image_features"
    source_manifest = json.loads((source / "image_features_manifest.json").read_text())
    require(
        source_manifest["model_id"] == "openai/clip-vit-base-patch32",
        "Unexpected image model.",
    )
    require(
        source_manifest["model_revision"] == "3d74acf9a28c67741b2f4f2ea7635f0aaf6f0268",
        "Unexpected CLIP revision.",
    )
    require(
        source_manifest["embedding_dim"] == 512
        and source_manifest["normalization"] == "L2",
        "Unexpected image vector contract.",
    )
    inputs = {"source_image_manifest": identity(source_manifest)}
    params = {
        "reuse_existing_clip": True,
        "catalog_filter": "C0",
        "normalization_tolerance": 0.002,
    }
    reused = reusable("retrievers/image", params, inputs, identity(catalog))
    if reused is not None:
        return reused
    local = new_stage("retrievers/image")
    staging = LOCAL_ROOT / "image_features"
    for filename, checksum in source_manifest["artifact_sha256"].items():
        copy_verified(source / filename, staging / filename, checksum)
    vectors = np.load(staging / "image_embeddings.npy", mmap_mode="r")
    mapping = pq.read_table(staging / "image_article_ids.parquet").sort_by(
        "embedding_row"
    )
    ids = mapping["article_id"].to_pylist()
    validate_mapping(ids, mapping["embedding_row"].to_pylist())
    article_catalog = {
        r[0] for r in db.execute("SELECT article_id FROM articles").fetchall()
    }
    require(set(ids) <= article_catalog, "Image mapping contains unknown catalog IDs.")
    require(
        pa.types.is_int64(mapping.schema.field("embedding_row").type),
        "Image embedding_row must be int64.",
    )
    require(
        vectors.shape == (len(ids), 512) and vectors.dtype == np.float32,
        "Image vector shape/dtype mismatch.",
    )
    require(
        len(ids) == source_manifest["embedded_article_count"],
        "Image manifest row mismatch.",
    )
    index = faiss.read_index(str(staging / "image_faiss.index"))
    require(
        isinstance(index, faiss.IndexFlatIP)
        and index.ntotal == len(ids)
        and index.d == 512,
        "Unexpected saved image index.",
    )
    for start in range(0, len(vectors), 4096):
        block = vectors[start : start + 4096]
        require(
            np.isfinite(block).all()
            and np.allclose(np.linalg.norm(block, axis=1), 1, atol=0.002),
            "Invalid image vectors.",
        )
        reconstructed = index.reconstruct_n(start, len(block))
        require(
            np.array_equal(block, reconstructed), "FAISS image-row alignment mismatch."
        )
    catalog_set = set(catalog)
    positions = [i for i, item in enumerate(ids) if item in catalog_set]
    require(bool(positions), "No eligible image embeddings.")
    filtered = np.lib.format.open_memmap(
        local / "vectors.npy", mode="w+", dtype=np.float32, shape=(len(positions), 512)
    )
    for start in range(0, len(positions), 4096):
        filtered[start : start + 4096] = vectors[positions[start : start + 4096]]
    filtered.flush()
    filtered_ids = [ids[i] for i in positions]
    pq.write_table(
        pa.table(
            {"article_id": filtered_ids, "embedding_row": range(len(filtered_ids))}
        ),
        local / "items.parquet",
    )
    filtered_index = flat_index(filtered)
    faiss.write_index(filtered_index, str(local / "index.faiss"))
    del index, vectors, filtered, filtered_index
    manifest = publish(
        "retrievers/image", local, params, inputs, identity(catalog), len(filtered_ids)
    )
    return local, manifest


def run_cpu_retrievers():
    import faiss
    from scipy import sparse

    started = time.perf_counter()
    faiss.omp_set_num_threads(THREADS)
    db, catalog, processed_manifest, protocol_manifest = open_context()
    parameters = {
        channel: cpu_candidate_parameters(channel)
        for channel in ("popularity", "itemcf", "als", "image")
    }
    model_started = time.perf_counter()
    fallback, popularity_manifest, cf_artifact, als_artifact = fit_cpu_models(
        db, catalog, processed_manifest
    )
    performance_event("all_cpu_model_preparation", model_started)
    fallback_compact = (
        np.searchsorted(catalog, [i for i, _ in fallback]).astype(np.int32),
        np.asarray([s for _, s in fallback], dtype=np.float32),
    )
    for info in anchor_schedule():
        roster = anchor_roster(db, info["anchor_date"])
        if pending_cpu_anchor(
            db,
            roster,
            catalog,
            "popularity",
            info,
            popularity_manifest,
            protocol_manifest,
            parameters["popularity"],
        ):
            emit_cpu_anchor(
                db,
                roster,
                catalog,
                "popularity",
                info["anchor_date"],
                info["split"],
                popularity_manifest,
                protocol_manifest,
                parameters["popularity"],
                popularity=fallback,
            )
    als_cache = None
    for info in anchor_schedule():
        roster = anchor_roster(db, info["anchor_date"])
        if not pending_cpu_anchor(
            db,
            roster,
            catalog,
            "als",
            info,
            als_artifact[1],
            protocol_manifest,
            parameters["als"],
        ):
            continue
        if als_cache is None:
            als_cache = prepare_frozen_als(
                db, als_artifact[0], catalog, fallback_compact
            )
        model_rows = als_anchor_rows(als_cache, roster)
        emit_cpu_anchor(
            db,
            roster,
            catalog,
            "als",
            info["anchor_date"],
            info["split"],
            als_artifact[1],
            protocol_manifest,
            parameters["als"],
            compute=lambda lo, hi, als_cache=als_cache, model_rows=model_rows: (
                frozen_als_results(als_cache, model_rows[lo:hi])
            ),
        )
    if als_cache is not None:
        performance_event(
            "als_cache_summary",
            started,
            hits=als_cache["hits"],
            searched_users=als_cache["misses"],
            search_seconds=als_cache["search_seconds"],
        )
        als_cache = None
    image_artifact = prepare_image_model(db, catalog)
    neighbors = vectors = image_index = cf_mapping = image_mapping = None
    for info in anchor_schedule():
        anchor = info["anchor_date"]
        roster = anchor_roster(db, anchor)
        cf_pending = pending_cpu_anchor(
            db,
            roster,
            catalog,
            "itemcf",
            info,
            cf_artifact[1],
            protocol_manifest,
            parameters["itemcf"],
        )
        image_pending = pending_cpu_anchor(
            db,
            roster,
            catalog,
            "image",
            info,
            image_artifact[1],
            protocol_manifest,
            parameters["image"],
        )
        if not (cf_pending or image_pending):
            continue
        history = prepare_anchor_history(db, roster, anchor)
        if cf_pending:
            if neighbors is None:
                neighbors = sparse.load_npz(cf_artifact[0] / "neighbors.npz")
                cf_ids = pq.read_table(cf_artifact[0] / "items.parquet")[
                    "article_id"
                ].to_pylist()
                require(cf_ids == catalog, "ItemCF model/catalog map mismatch.")
                cf_mapping = source_item_mapping(db, cf_ids)
            emit_cpu_anchor(
                db,
                roster,
                catalog,
                "itemcf",
                anchor,
                info["split"],
                cf_artifact[1],
                protocol_manifest,
                parameters["itemcf"],
                history_seconds=history["seconds"],
                compute=partial(
                    compact_cf_results,
                    history,
                    source_to_model=cf_mapping,
                    neighbors=neighbors,
                    fallback=fallback_compact,
                ),
            )
        if image_pending:
            if vectors is None:
                vectors = np.load(image_artifact[0] / "vectors.npy", mmap_mode="r")
                if (
                    MODE == "full"
                    and vectors.nbytes < CPU_RESOURCES["usable_ram_bytes"] / 10
                ):
                    vectors = np.asarray(vectors).copy()
                image_ids = pq.read_table(image_artifact[0] / "items.parquet")[
                    "article_id"
                ].to_pylist()
                require(
                    len(image_ids) == len(set(image_ids)), "Duplicate image model IDs."
                )
                image_mapping = source_item_mapping(db, image_ids)
                image_order = np.argsort(image_ids)
                image_positions = np.searchsorted(catalog, image_ids).astype(np.int32)
                image_index = faiss.read_index(str(image_artifact[0] / "index.faiss"))

            def image_compute(
                lo,
                hi,
                history=history,
                image_mapping=image_mapping,
                vectors=vectors,
                image_index=image_index,
                image_ids=image_ids,
                image_positions=image_positions,
                image_order=image_order,
            ):
                preferences, valid = compact_image_preferences(
                    history, lo, hi, image_mapping, vectors
                )
                return compact_index_results(
                    image_index,
                    preferences,
                    valid,
                    image_ids,
                    image_positions,
                    fallback_compact,
                    model_order=image_order,
                )

            emit_cpu_anchor(
                db,
                roster,
                catalog,
                "image",
                anchor,
                info["split"],
                image_artifact[1],
                protocol_manifest,
                parameters["image"],
                compute=image_compute,
                history_seconds=history["seconds"],
            )
        history = None
    db.close()
    performance_event("cpu_pipeline_total", started)
    profile_id = uuid.uuid4().hex[:12]
    stage = f"retrievers/cpu_performance/{profile_id}"
    local = new_stage(stage)
    write_json(local / "performance.json", CPU_PERFORMANCE)
    publish(
        stage,
        local,
        {"implementation": "cpu-execution-2", "resources": CPU_RESOURCES},
        {"processed_manifest": identity(processed_manifest)},
        identity(catalog),
        len(CPU_PERFORMANCE),
    )
    print(
        "CPU retrieval complete. Full-scale performance is measured only by this run."
    )

## Execute this stage
This is the entry point. It requires the documented completed prerequisites. No models or datasets are downloaded during local source validation.

### CPU execution and restart behavior

Full mode detects CPU affinity/cgroup quotas and available/container RAM. On a 48-core,
176 GiB runtime it selects up to 16 workers, 32 GB DuckDB memory and 4096-query
batches. Smaller runtimes are clamped. ALS BLAS uses one thread. DuckDB can spill.
Candidate Parquet files stream by anchor toward 192 MiB compressed shards in full
mode (8 MiB in smoke); a final shard may be smaller and a batch can overshoot.

Popularity cross-joins the frozen Top-100 without user histories. ALS caches frozen
Top-100 results for frequently repeated fitted users within a RAM budget. ItemCF
and image retrieval share one strictly pre-anchor history aggregation; the original
100-item limit applies before model eligibility filtering. Exact inner-product FAISS,
weighted binary-cosine ItemCF, recency weighting and tie-breaking are retained.

Completed stages are reused only after manifest, input, schema and checksum checks.
The existing `model_parameters` execution-size fields remain compatibility descriptors
for readers 03/04: they retain the legacy values (or the values already published
for that channel). Actual resource policy is recorded separately under `execution`;
actual ItemCF block sizes appear in performance diagnostics. No global namespace,
model cutoff, query population or candidate schema changes are introduced.

Incomplete candidate attempts use unique `shard-*.parquet` filenames; only the new
validated files enter the completion manifest. Old remote attempt files remain
untouched. Readers stage only manifest-declared files. Conflicting partial model
files fail with a manual-review message; completed incompatible or corrupt stages
also fail rather than overwrite data. Do not run concurrent publishers into one stage.

Per-anchor diagnostics report history, compute, write, validation and publication
times, throughput, peak process RSS, file counts and bytes. A separate uniquely named
CPU performance artifact captures the session summary. Reused stages are identified
explicitly. Actual Full Colab runtime, memory and file-count improvements are **not yet
measured** by local synthetic verification. No approximate retrieval is introduced;
floating-point kernels may still vary slightly across hardware/library versions.


In [6]:
runtime_start()
metric_smoke_checks()
run_cpu_retrievers()

Mounted at /content/drive
{'mode': 'full', 'resources': {'threads': 16, 'memory_limit': '32.00GB', 'query_batch_size': 4096, 'target_shard_bytes': 201326592, 'als_cache_bytes': 1073741824, 'sparse_workspace_bytes': 2147483648, 'usable_ram_bytes': 186599010304, 'available_cpus': 48}, 'result_scope': 'full protocol', 'local_free_GiB': 235.4}
{"operation": "all_cpu_model_preparation", "process_peak_RSS_GiB": 0.25438690185546875, "seconds": 14.971032529000013}
{"copied_candidate_files": 0, "files": 316, "operation": "verified_completed_anchor", "process_peak_RSS_GiB": 0.32543182373046875, "rows": 8071800, "seconds": 127.64603319100001, "stage": "candidates/ranking_train/popularity/2020-07-28"}
{"copied_candidate_files": 0, "files": 321, "operation": "verified_completed_anchor", "process_peak_RSS_GiB": 0.37090301513671875, "rows": 8205900, "seconds": 126.89759385000002, "stage": "candidates/ranking_train/popularity/2020-07-31"}
{"copied_candidate_files": 0, "files": 310, "operation": "verif

# Data Validation

In [1]:
from pathlib import Path
from collections import defaultdict
import hashlib
import json
import time

import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq

from google.colab import drive

drive.mount("/content/drive")

DATA_ROOT = Path(
    "/content/drive/MyDrive/projects/e_commerce_recsys/data"
)

MODE = "full"
NAMESPACE = "mvp_v1"

CHANNELS = (
    "popularity",
    "itemcf",
    "als",
    "image",
)

VERSION = "hm-mvp-1.0.0"
PROTOCOL_VERSION = "1.0"

SCHEDULE = {
    "ranking_train": (
        "2020-07-28",
        "2020-07-31",
        "2020-08-03",
        "2020-08-06",
        "2020-08-09",
    ),
    "validation": (
        "2020-08-16",
        "2020-08-19",
        "2020-08-22",
        "2020-08-25",
        "2020-08-28",
    ),
    "test": (
        "2020-09-04",
        "2020-09-07",
        "2020-09-10",
        "2020-09-13",
        "2020-09-16",
    ),
}

CANDIDATE_SCHEMA = pa.schema([
    ("query_id", pa.string()),
    ("customer_id", pa.string()),
    ("anchor_date", pa.date32()),
    ("article_id", pa.string()),
    ("channel", pa.string()),
    ("raw_score", pa.float32()),
    ("source_rank", pa.int32()),
    ("retriever_version", pa.string()),
])


def require(condition, message):
    if not condition:
        raise AssertionError(message)


def stage_dir(stage):
    return DATA_ROOT / stage / NAMESPACE / MODE


def manifest_id(manifest):
    payload = json.dumps(
        manifest,
        sort_keys=True,
        separators=(",", ":"),
        default=str,
    )
    return hashlib.sha256(payload.encode()).hexdigest()


def sha256_file(path):
    digest = hashlib.sha256()

    with path.open("rb") as file:
        for chunk in iter(
            lambda: file.read(8 * 1024 * 1024),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


def verify_stage(stage, expected_catalog=None):
    folder = stage_dir(stage)
    marker = folder / "_COMPLETE.json"

    require(
        marker.is_file(),
        f"Missing completion marker: {marker}",
    )

    manifest = json.loads(marker.read_text())

    require(
        manifest.get("stage") == stage,
        f"Incorrect stage identity: {stage}",
    )

    require(
        manifest.get("mode") == MODE,
        f"Wrong execution mode: {stage}",
    )

    require(
        manifest.get("artifact_version") == VERSION,
        f"Wrong artifact version: {stage}",
    )

    require(
        manifest.get("protocol_version") == PROTOCOL_VERSION,
        f"Wrong protocol version: {stage}",
    )

    require(
        manifest.get("validation_status") == "validated",
        f"Stage not validated: {stage}",
    )

    if expected_catalog is not None:
        require(
            manifest.get("catalog_id") == expected_catalog,
            f"Catalog mismatch: {stage}",
        )

    files = manifest.get("files")

    require(
        isinstance(files, dict) and bool(files),
        f"Empty file manifest: {stage}",
    )

    require(
        set(files) == set(manifest.get("output_paths", [])),
        f"Output list mismatch: {stage}",
    )

    for filename, info in files.items():
        relative_path = Path(filename)

        require(
            not relative_path.is_absolute()
            and ".." not in relative_path.parts,
            f"Unsafe artifact path: {filename}",
        )

        path = folder / relative_path

        require(
            path.is_file(),
            f"Missing artifact: {path}",
        )

        require(
            path.stat().st_size == info["bytes"],
            f"File size mismatch: {path}",
        )

        require(
            sha256_file(path) == info["sha256"],
            f"SHA256 mismatch: {path}",
        )

    return manifest


started = time.perf_counter()

# Step 1: Verify canonical data and protocol
print("Checking canonical data and protocol...")

processed = verify_stage("processed")

protocol = verify_stage(
    "protocol",
    processed["catalog_id"],
)

require(
    protocol["input_fingerprints"].get(
        "processed_manifest"
    ) == manifest_id(processed),
    "Protocol references an incompatible dataset.",
)

protocol_dir = stage_dir("protocol")

catalog = pq.read_table(
    protocol_dir / "eligible_catalog.parquet",
    columns=["article_id"],
)["article_id"].to_pylist()

require(
    catalog == sorted(set(catalog)),
    "Eligible catalog is not sorted and unique.",
)

require(
    all(
        len(article_id) == 10 and article_id.isdigit()
        for article_id in catalog
    ),
    "Invalid article IDs in catalog.",
)

catalog_id = manifest_id(catalog)

require(
    catalog_id == protocol["catalog_id"],
    "Catalog content does not match catalog ID.",
)

print(f"Canonical data: PASS")
print(f"Protocol: PASS")
print(f"Eligible catalog articles: {len(catalog):,}")


# Step 2: Verify frozen CPU retriever models
print("\nChecking CPU retriever models...")

models = {}

for channel in CHANNELS:
    models[channel] = verify_stage(
        f"retrievers/{channel}",
        catalog_id,
    )

    if channel in ("popularity", "itemcf", "als"):
        require(
            models[channel]["model_parameters"].get("cutoff")
            == "2020-07-28",
            f"Incorrect training cutoff: {channel}",
        )

    print(f"PASS: {channel}")


# Step 3: Verify all persisted candidate stages
print("\nChecking all 60 candidate stages...")

summary = defaultdict(
    lambda: {
        "anchors": 0,
        "queries": 0,
        "rows": 0,
        "shards": 0,
    }
)

channel_parameters = {}

for split, anchors in SCHEDULE.items():
    for anchor in anchors:

        roster_path = (
            protocol_dir
            / "queries"
            / f"{split}_{anchor}"
            / "queries.parquet"
        )

        require(
            roster_path.is_file(),
            f"Missing query roster: {roster_path}",
        )

        roster = pq.read_table(
            roster_path,
            columns=["eligible"],
        )

        query_count = int(
            pc.sum(roster["eligible"]).as_py() or 0
        )

        require(
            query_count > 0,
            f"No eligible queries: {split}/{anchor}",
        )

        for channel in CHANNELS:

            stage = (
                f"candidates/{split}/{channel}/{anchor}"
            )

            manifest = verify_stage(
                stage,
                catalog_id,
            )

            require(
                manifest.get("snapshot_or_anchor") == anchor,
                f"Anchor mismatch: {stage}",
            )

            expected_inputs = {
                "model_manifest": manifest_id(
                    models[channel]
                ),
                "protocol_manifest": manifest_id(
                    protocol
                ),
            }

            require(
                manifest.get("input_fingerprints")
                == expected_inputs,
                f"Artifact lineage mismatch: {stage}",
            )

            parameters = manifest.get(
                "model_parameters", {}
            )

            require(
                parameters.get("channel") == channel,
                f"Channel configuration mismatch: {stage}",
            )

            if channel in channel_parameters:
                require(
                    parameters == channel_parameters[channel],
                    f"Inconsistent parameters: {stage}",
                )
            else:
                channel_parameters[channel] = parameters

            shard_names = sorted(
                name
                for name in manifest["files"]
                if name.startswith("shard-")
                and name.endswith(".parquet")
            )

            require(
                bool(shard_names),
                f"No Parquet candidate files: {stage}",
            )

            rows = 0

            for filename in shard_names:
                path = stage_dir(stage) / filename
                parquet = pq.ParquetFile(path)

                require(
                    parquet.schema_arrow
                    .remove_metadata()
                    .equals(CANDIDATE_SCHEMA),
                    f"Candidate schema mismatch: {path}",
                )

                rows += parquet.metadata.num_rows

            require(
                rows == manifest["row_count"],
                f"Candidate row count mismatch: {stage}",
            )

            require(
                query_count <= rows <= query_count * 100,
                f"Invalid candidate count: {stage}",
            )

            if channel in ("popularity", "als"):
                require(
                    rows == query_count * 100,
                    f"Expected 100 candidates per query: {stage}",
                )

            stats = summary[channel]

            stats["anchors"] += 1
            stats["queries"] += query_count
            stats["rows"] += rows
            stats["shards"] += len(shard_names)

            print(
                f"PASS | {split:13s} | {anchor} | "
                f"{channel:10s} | "
                f"queries={query_count:,} | "
                f"rows={rows:,} | "
                f"shards={len(shard_names)}",
                flush=True,
            )


# Step 4: Final consistency checks
require(
    all(
        stats["anchors"] == 15
        for stats in summary.values()
    ),
    "One or more retrieval channels are incomplete.",
)

print("\n" + "=" * 60)
print("ALL CHECKS PASSED")
print("=" * 60)

for channel, stats in summary.items():
    print(
        f"{channel:10s} | "
        f"anchors={stats['anchors']:2d} | "
        f"queries={stats['queries']:,} | "
        f"candidates={stats['rows']:,} | "
        f"shards={stats['shards']:,}"
    )

elapsed_minutes = (
    time.perf_counter() - started
) / 60

print(f"\nVerification time: {elapsed_minutes:.2f} minutes")
print("Canonical data: VERIFIED")
print("Evaluation protocol: VERIFIED")
print("CPU retriever models: VERIFIED")
print("CPU candidate stages: 60 / 60 VERIFIED")
print("Ready for downstream artifact loading.")

Mounted at /content/drive
Checking canonical data and protocol...
Canonical data: PASS
Protocol: PASS
Eligible catalog articles: 98,085

Checking CPU retriever models...
PASS: popularity
PASS: itemcf
PASS: als
PASS: image

Checking all 60 candidate stages...
PASS | ranking_train | 2020-07-28 | popularity | queries=80,718 | rows=8,071,800 | shards=316
PASS | ranking_train | 2020-07-28 | itemcf     | queries=80,718 | rows=8,029,391 | shards=316
PASS | ranking_train | 2020-07-28 | als        | queries=80,718 | rows=8,071,800 | shards=1
PASS | ranking_train | 2020-07-28 | image      | queries=80,718 | rows=8,071,725 | shards=1
PASS | ranking_train | 2020-07-31 | popularity | queries=82,059 | rows=8,205,900 | shards=321
PASS | ranking_train | 2020-07-31 | itemcf     | queries=82,059 | rows=8,161,594 | shards=1
PASS | ranking_train | 2020-07-31 | als        | queries=82,059 | rows=8,205,900 | shards=1
PASS | ranking_train | 2020-07-31 | image      | queries=82,059 | rows=8,205,834 | shards=1